# Results Gate Policy

Purpose: tune edge-floor and side-specific policy thresholds, then pick an operating point for live betting.

Use this notebook to answer:
- What edge floor produces stable ROI + CLV?
- Should over/under use different floors?
- Is the gate helping enough to justify stricter filtering?

Edge-floor and recommendation-mix simulator for BET/HOLD policy tuning.

In [1]:
from pathlib import Path
import subprocess
import sys
import polars as pl
from IPython.display import HTML, display

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "production").exists() and (candidate / "src" / "Python").exists():
        ROOT = candidate
        break

sys.path.insert(0, str(ROOT / "src"))
from Python.notebook_analysis_utils import has_over_clv_red_flag

SIM = ROOT / "production" / "ops" / "policy_simulator.py"
OUT_SWEEP = ROOT / "artifacts" / "odds_log" / "policy_scenario_sweep.parquet"


def show_table(df: pl.DataFrame, max_rows: int = 30, height: int = 420):
    pdf = df.to_pandas().round(3)
    if len(pdf) <= max_rows:
        display(pdf)
        return
    table = pdf.to_html(index=False, na_rep="—")
    display(HTML(f"<div style='max-height:{height}px; overflow:auto; border:1px solid #4443; border-radius:6px'>{table}</div>"))


print("repo:", ROOT)
print("simulator:", SIM)

repo: C:\Users\ckaplinger\Downloads\Personal-Projects\MLB-Props
simulator: C:\Users\ckaplinger\Downloads\Personal-Projects\MLB-Props\production\ops\policy_simulator.py


In [2]:
thresholds = "0.08,0.10,0.12,0.14,0.16,0.18"
cmd = [sys.executable, str(SIM), "--thresholds", thresholds]
run = subprocess.run(cmd, cwd=str(ROOT), capture_output=True, text=True)
print(run.stdout)
if run.returncode != 0:
    raise RuntimeError(run.stderr)

--- latest recommendation mix ---
{'recommendation': 'skip', 'oos_reason': None, 'n': 3}
{'recommendation': 'HOLD', 'oos_reason': None, 'n': 1}
{'recommendation': 'BET', 'oos_reason': None, 'n': 1}
wrote C:\Users\ckaplinger\Downloads\Personal-Projects\MLB-Props\artifacts\odds_log\policy_scenario_sweep.parquet
wrote C:\Users\ckaplinger\Downloads\Personal-Projects\MLB-Props\artifacts\odds_log\policy_scenario_sweep_latest.csv
--- latest scenario rows ---
{'snapshot_utc': '2026-09-11T01:20:43.005294+00:00', 'scope': 'all', 'edge_floor': 0.08, 'n_bets': 328, 'wins': 162, 'losses': 166, 'win_rate': 0.49390243902439024, 'total_pnl': 190.20423133053896, 'roi': 0.008129376207018084, 'avg_edge': 0.16614601736091666, 'avg_clv_pp': 0.006596134263280783, 'total_stake': 23397.149607412168}
{'snapshot_utc': '2026-09-11T01:20:43.010376+00:00', 'scope': 'all', 'edge_floor': 0.1, 'n_bets': 287, 'wins': 144, 'losses': 143, 'win_rate': 0.5017421602787456, 'total_pnl': 480.4226225671935, 'roi': 0.022001976

In [3]:
if not OUT_SWEEP.exists():
    print("No scenario sweep artifact yet.")
else:
    sweep = pl.read_parquet(OUT_SWEEP)
    latest_ts = sweep.select(pl.col("snapshot_utc").max()).item()
    latest = sweep.filter(pl.col("snapshot_utc") == latest_ts)
    latest_view = latest.sort(["scope", "edge_floor"])
    if "show_table" in globals():
        show_table(latest_view)
    else:
        print(latest_view)

    if "scope" in latest.columns and "n_bets" in latest.columns:
        all_rows = latest.filter(pl.col("scope") == "all")
        if all_rows.height:
            raw_n = int(all_rows.sort("edge_floor").head(1).select("n_bets").item())
            best_n = int(all_rows.sort("edge_floor", descending=True).head(1).select("n_bets").item())
            print(f"raw_n(at lowest floor)={raw_n} best_line_like_n(at highest floor)={best_n}")

    side_view = latest.filter(pl.col("scope").is_in(["over", "under"])) if "scope" in latest.columns else pl.DataFrame()
    if side_view.height:
        side_health = side_view.select([c for c in ["scope", "edge_floor", "roi", "avg_clv_pp", "n_bets"] if c in side_view.columns]).sort(["scope", "edge_floor"])
        print("\nside CLV/ROI health")
        print(side_health)
        side_health_check = side_view.rename({"scope": "side", "avg_clv_pp": "mean_clv_pp"})
        if has_over_clv_red_flag(side_health_check):
            print("RED FLAG: over avg_clv_pp <= 0 for one or more thresholds.")

,snapshot_utc,scope,edge_floor,n_bets,wins,losses,win_rate,total_pnl,roi,avg_edge,avg_clv_pp,total_stake,edge_floor_over,edge_floor_under
0,2026-09-11T01:20:43.061154+00:00,under,0.18,58,35,23,0.603,1490.271,0.236,0.242,0.014,6313.871,NaN,NaN



side CLV/ROI health
shape: (1, 5)
┌───────┬────────────┬──────────┬────────────┬────────┐
│ scope ┆ edge_floor ┆ roi      ┆ avg_clv_pp ┆ n_bets │
│ ---   ┆ ---        ┆ ---      ┆ ---        ┆ ---    │
│ str   ┆ f64        ┆ f64      ┆ f64        ┆ i64    │
╞═══════╪════════════╪══════════╪════════════╪════════╡
│ under ┆ 0.18       ┆ 0.236031 ┆ 0.013718   ┆ 58     │
└───────┴────────────┴──────────┴────────────┴────────┘


In [4]:
# Governance replay risk panel (policy context, not feature bakeoff)
REPLAY_PATH = ROOT / "artifacts" / "odds_log" / "policy_replay_daily.json"

if not REPLAY_PATH.exists():
    print(f"Missing {REPLAY_PATH}. Run production/ops/build_policy_governance_report.py")
else:
    import json

    replay = json.loads(REPLAY_PATH.read_text(encoding="utf-8"))
    scenarios = replay.get("scenarios", []) if isinstance(replay.get("scenarios"), list) else []
    if not scenarios:
        print("No replay scenarios found.")
    else:
        risk_df = pl.DataFrame(scenarios)
        keep = [
            "scenario",
            "n",
            "roi",
            "clv_mean_pp",
            "geo_growth_log_mean",
            "mc_prob_bankroll_floor_breach",
            "mc_prob_drawdown_breach",
            "mc_median_terminal_bankroll",
            "mc_p10_terminal_bankroll",
        ]
        risk_view = risk_df.select([c for c in keep if c in risk_df.columns]).sort("scenario")
        print("Policy replay risk panel")
        show_table(risk_view, max_rows=20)

Policy replay risk panel


,scenario,n,roi,clv_mean_pp,geo_growth_log_mean,mc_prob_bankroll_floor_breach,mc_prob_drawdown_breach,mc_median_terminal_bankroll,mc_p10_terminal_bankroll
0,current_policy_1p5u_over_15pct,218,0.023,0.007,0.0,0.0,0.002,1.055,0.779
1,current_policy_2u_over_18pct,218,0.029,0.007,0.0,0.0,0.014,1.072,0.751
2,current_policy_flat_1u,218,0.015,0.007,0.0,0.0,0.000,1.022,0.830
3,historical_actual_stake,297,0.015,0.006,0.0,0.0,0.012,1.011,0.731


In [5]:
# Regime-aware note: recent windows vs full history
# This controls for production changes in features/calibration/edge floors.
LEDGER_PATH = ROOT / "artifacts" / "odds_log" / "ledger.parquet"
if not LEDGER_PATH.exists():
    print(f"Missing {LEDGER_PATH}")
else:
    led = pl.read_parquet(LEDGER_PATH)
    settled = led.filter(
        (pl.col("status") == "settled")
        & (pl.col("stake").cast(pl.Float64).fill_null(0) > 0)
        & pl.col("edge").is_not_null()
    ).with_columns(
        pl.col("game_date").cast(pl.Utf8).str.slice(0, 10).alias("gdate")
    )

    if settled.is_empty():
        print("No settled rows with stake>0.")
    else:
        latest = settled.select(pl.col("gdate").max()).item()
        windows = [
            ("full_history", None),
            ("last_60_settled", 60),
            ("last_30_settled", 30),
        ]
        rows = []
        sorted_settled = settled.sort("gdate")
        for label, n in windows:
            scope = sorted_settled if n is None else sorted_settled.tail(n)
            stake = float(scope["stake"].cast(pl.Float64).sum())
            pnl = float(scope["pnl"].cast(pl.Float64).sum())
            clv = float(scope["clv_pp"].cast(pl.Float64).mean()) if "clv_pp" in scope.columns and scope.height else None
            rows.append({
                "window": label,
                "asof_game_date": latest,
                "n": int(scope.height),
                "stake": stake,
                "pnl": pnl,
                "roi": (pnl / stake) if stake > 0 else None,
                "mean_clv_pp": clv,
            })
        print("Regime-aware aggregate check (interpret full-history with caution):")
        show_table(pl.DataFrame(rows), max_rows=10)

Regime-aware aggregate check (interpret full-history with caution):


,window,asof_game_date,n,stake,pnl,roi,mean_clv_pp
0,full_history,2026-09-09,523,37891.909,766.548,0.020,0.004
1,last_60_settled,2026-09-09,60,4489.020,646.268,0.144,-0.005
2,last_30_settled,2026-09-09,30,1953.847,333.654,0.171,-0.004


In [6]:
# Policy edge-decile realization (current settled policy context)
if not LEDGER_PATH.exists():
    print(f"Missing {LEDGER_PATH}")
else:
    led = pl.read_parquet(LEDGER_PATH)
    settled = led.filter(
        (pl.col("status") == "settled")
        & (pl.col("stake").cast(pl.Float64).fill_null(0) > 0)
        & pl.col("edge").is_not_null()
    ).with_columns(
        pl.col("edge").cast(pl.Float64).alias("edge_f")
    )
    if settled.height < 20:
        print("Not enough settled rows for decile analysis.")
    else:
        dec = (
            settled.with_columns(
                pl.col("edge_f")
                .rank(method="ordinal")
                .mul(10)
                .truediv(pl.lit(float(settled.height)))
                .ceil()
                .clip(1, 10)
                .cast(pl.Int64)
                .alias("edge_decile")
            )
            .group_by("edge_decile")
            .agg(
                pl.len().alias("n"),
                pl.col("stake").cast(pl.Float64).sum().alias("stake"),
                pl.col("pnl").cast(pl.Float64).sum().alias("pnl"),
                pl.col("edge_f").mean().alias("mean_edge"),
                pl.col("clv_pp").cast(pl.Float64).mean().alias("mean_clv_pp"),
            )
            .with_columns(
                pl.when(pl.col("stake") > 0).then(pl.col("pnl") / pl.col("stake")).otherwise(None).alias("roi")
            )
            .sort("edge_decile")
        )
        print("Edge-decile realization (policy context)")
        show_table(dec, max_rows=20)

Edge-decile realization (policy context)


,edge_decile,n,stake,pnl,mean_edge,mean_clv_pp,roi
0,1,52,1949.305,-385.964,0.089,0.008,-0.198
1,2,52,2594.021,-671.592,0.110,0.001,-0.259
2,3,52,2638.455,583.480,0.124,0.007,0.221
3,4,53,3036.786,-317.448,0.136,0.002,-0.105
4,5,52,3298.963,-350.725,0.149,-0.010,-0.106
5,6,52,3362.053,-253.121,0.163,0.000,-0.075
6,7,53,4163.148,-1172.111,0.182,0.007,-0.282
7,8,52,4641.410,44.500,0.201,0.002,0.010
8,9,52,5208.663,1475.677,0.227,0.011,0.283
9,10,53,6999.106,1813.851,0.299,0.006,0.259
